# Solutions: Continuous Integration in RAP pipeline

This notebook provides step-by-step solutions for implementing and testing continuous integration (CI) best practices in your RAP pipeline. Each solution matches the corresponding exercise notebook and includes instructions for checking your changes and expected results.

## Solution 1: Explore the CI configuration files

- Open `.pre-commit-config.yaml`, `pyproject.toml`, and `requirements.txt` in your project root.
- Review which hooks and tools are enabled for Python scripts and notebooks.
- **Expected result:** You should see hooks for black, isort, flake8, nbQA, detect-secrets, and more. Notebook hooks use `nbqa-black` and `nbqa-isort`.

## Solution 2: Add a new pre-commit hook and test with a deliberate error

- Add the following to `.pre-commit-config.yaml` under the `pre-commit-hooks` repo:
  ```yaml
  - id: check-ast
    name: Check Python files for syntax errors
    exclude: ^docs/user_guide/user_documentation\.md$
  ```
- Run the following commands in your terminal:
  ```cmd
  pre-commit install
  pre-commit run --all-files
  ```
- Add a deliberate syntax error to any Python file (e.g., `def broken_func(`).
- Run the hook again:
  ```cmd
  pre-commit run check-ast --all-files
  ```
- **How to check:** The hook will fail and report the syntax error, e.g.:
  ```
  File "src/python_rap_demo/cleaning.py", line X
    def broken_func(
                   ^
  SyntaxError: unexpected EOF while parsing
  ```
- Fix the error and rerun the hook to confirm it passes.
- **Expected result:**
  - After adding the error: The hook fails and reports the syntax error.
  - After fixing the error: The hook passes with no errors.

## Solution 3: Detect secrets in your code

- Add a fake secret to any Python file. For example:  

  ```python
  password = "12345"
  print(password)
  ```
- Run:
  ```cmd
  pre-commit run bandit --all-files
  ```
- **How to check:** The hook will flag the secret in the output. Remove the secret and rerun the hook.
- **Expected result:** No secrets detected after removal; hook passes.

## Solution 4: Add a new workflow that activates on push to any branch

- Create a new workflow file in `.github/workflows/ci_lint_and_format_any_branch.yml` with the following content:
  ```yaml
  # This workflow runs linting and formatting checks on every branch push
  name: Lint and Format CI (Any Branch)
  on:
    push:
      branches: [ "*" ]  # Trigger on push to any branch
  jobs:
    lint-and-format:
      runs-on: ubuntu-latest  # Use the latest Ubuntu runner
      steps:
        - uses: actions/checkout@v4  # Check out the repository code
        - name: Set up Python
          uses: actions/setup-python@v5
          with:
            python-version: '3.11'  # Use Python 3.11
        - name: Install dependencies
          run: |
            python -m pip install --upgrade pip  # Upgrade pip
            pip install -r requirements.txt      # Install required packages
        - name: Run Black (code formatter)
          run: |
            black --check src/ tests/  # Check code formatting in src/ and tests/
        - name: Run Flake8 (linter)
          run: |
            flake8 src/ tests/         # Run linting in src/ and tests/
  ```
- Commit and push your change to any branch.
- Go to your repository on GitHub and click the **Actions** tab at the top of the page.
- Look for the workflow run named "Lint and Format CI (Any Branch)" in the list of workflow runs.
- Click on the workflow run to view its status and logs.
- **Expected result:** The workflow should run automatically on any branch push and show its status (success or failure) in the Actions tab. If there are linting or formatting errors, they will be shown in the logs and the workflow will fail until fixed.

This action will run any time you push code up to GitHub on any branch.

## Reflection

By following these solutions, you have implemented and tested continuous integration in your RAP pipeline. This ensures your code is reproducible, robust, and meets quality standards automatically.